# S03E02 — Firmware

**Zadanie:** Uruchomić plik `/opt/firmware/cooler/cooler.bin` na wirtualnej maszynie Linux  
przez shell API. Zdobyć hasło, skonfigurować `settings.ini`, uruchomić binarkę → odebrać kod ECCS.

**Zasady:** tylko konto zwykłego użytkownika, nie wolno czytać `/etc`, `/root`, `/proc/`,  
respektować pliki `.gitignore`.

**Techniki:** agent z function calling, exploracja systemu plików, analiza błędów

In [ ]:
import os, json, time, re, requests
from anthropic import Anthropic
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

API_KEY    = os.getenv("AI_DEVS_API_KEY")
client     = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
VERIFY_URL = "https://hub.ag3nts.org/verify"
SHELL_URL  = "https://hub.ag3nts.org/api/shell"

print("Konfiguracja OK.")

In [ ]:
def shell(cmd: str, retries: int = 5) -> str:
    """Wykonaj polecenie na wirtualnej maszynie przez shell API."""
    backoff = 3
    for attempt in range(retries):
        resp = requests.post(SHELL_URL, json={"apikey": API_KEY, "cmd": cmd})
        if resp.status_code == 200:
            try:
                data = resp.json()
                # API może zwrócić wynik w różnych polach
                output = data.get("output") or data.get("result") or data.get("message") or str(data)
                return output
            except Exception:
                return resp.text
        elif resp.status_code in (429, 503):
            wait = int(resp.headers.get("Retry-After", backoff))
            print(f"  {resp.status_code} — czekam {wait}s")
            time.sleep(wait)
            backoff = min(backoff * 2, 30)
        else:
            return f"HTTP {resp.status_code}: {resp.text[:200]}"
    return "Błąd: przekroczono limit prób"


# Sprawdź dostępne komendy
print(shell("help"))

In [ ]:
# Agent z function calling — exploruje system i uruchamia binarkę

TOOLS = [{
    "name": "run_shell",
    "description": "Wykonaj polecenie na wirtualnej maszynie Linux przez API powłoki.",
    "input_schema": {
        "type": "object",
        "properties": {
            "cmd": {"type": "string", "description": "Polecenie do wykonania"}
        },
        "required": ["cmd"]
    }
}]

SYSTEM = """Jesteś agentem eksplorującym wirtualną maszynę Linux przez API powłoki.

Twój cel: uruchomić /opt/firmware/cooler/cooler.bin i zdobyć kod ECCS-xxxx.

Kroki:
1. Sprawdź dostępne komendy (run_shell "help")
2. Sprawdź zawartość /opt/firmware/cooler/ (ls)
3. Spróbuj uruchomić cooler.bin
4. Jeśli wymaga hasła — znajdź je w systemie (szukaj w plikach konfiguracyjnych)
5. Jeśli wymaga konfiguracji — edytuj settings.ini
6. Uruchom binarkę i odczytaj kod ECCS-xxxx

Zasady bezpieczeństwa (MUSISZ przestrzegać, inaczej dostęp zostanie zablokowany):
- NIE zaglądaj do /etc, /root, /proc/
- Respektuj pliki .gitignore — nie dotykaj wymienionych w nich plików

Gdy znajdziesz kod ECCS-xxxx, odpowiedz: ZNALEZIONO: <kod>"""

messages = [{"role": "user", "content": "Zacznij od sprawdzenia dostępnych komend i pliku binarnego."}]
MAX_STEPS = 40
found_code = None

for step in range(MAX_STEPS):
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=SYSTEM,
        tools=TOOLS,
        messages=messages
    )

    # Zbierz tekst i wywołania narzędzi
    text_parts = []
    tool_calls = []
    for block in resp.content:
        if block.type == "text":
            text_parts.append(block.text)
        elif block.type == "tool_use":
            tool_calls.append(block)

    agent_text = " ".join(text_parts)
    if agent_text:
        print(f"\nKrok {step+1} Agent: {agent_text[:200]}")

    # Sprawdź czy agent znalazł kod
    eccs_match = re.search(r'ECCS-[A-Za-z0-9]+', agent_text)
    if eccs_match or "ZNALEZIONO" in agent_text:
        found_code = eccs_match.group(0) if eccs_match else None
        print(f"\nKOD ZNALEZIONY: {found_code}")
        break

    if not tool_calls:
        print("Agent zakończył bez kodu — sprawdź output powyżej.")
        break

    # Wykonaj wywołania narzędzi i zbierz wyniki
    messages.append({"role": "assistant", "content": resp.content})
    tool_results = []

    for tc in tool_calls:
        cmd    = tc.input.get("cmd", "")
        output = shell(cmd)
        print(f"  $ {cmd}")
        print(f"    {output[:300]}")

        # Sprawdź wynik polecenia na kod ECCS
        if re.search(r'ECCS-[A-Za-z0-9]+', output):
            found_code = re.search(r'ECCS-[A-Za-z0-9]+', output).group(0)
            print(f"\nKOD W WYNIKU KOMENDY: {found_code}")

        tool_results.append({
            "type": "tool_result",
            "tool_use_id": tc.id,
            "content": output
        })

    messages.append({"role": "user", "content": tool_results})

    if found_code:
        break

print(f"\nKońcowy kod: {found_code}")

In [ ]:
assert found_code, "Nie znaleziono kodu ECCS — przejrzyj kroki agenta powyżej."

result = requests.post(VERIFY_URL, json={
    "apikey": API_KEY,
    "task": "firmware",
    "answer": {"confirmation": found_code}
}).json()

print("Odpowiedź Hub-u:", result)